In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')


# CONFIGURATION
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("=" * 70)
print("ENHANCED SIMPLE MODEL - UPGRADED WITH XGBOOST, LGBM, CATBOOST")
print("=" * 70)

# =============================================================================
# Load Data
# =============================================================================
print("\n[1/8] Loading Data...")
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')
submission = pd.read_csv('sample_submissions.csv')

print(f"Train shape: {train_data.shape}")
print(f"Test shape: {test_data.shape}")
print(f"Target distribution: {dict(train_data['status_gagal_bayar'].value_counts())}")

# =============================================================================
# PREPROCESSING 
# =============================================================================

train_data['is_train'] = 1
test_data['is_train'] = 0
test_data['status_gagal_bayar'] = np.nan

full_data = pd.concat([train_data, test_data], ignore_index=True)

# Date features (same as original)
full_data['tanggal_pencairan'] = pd.to_datetime(full_data['tanggal_pencairan'])
full_data['tahun'] = full_data['tanggal_pencairan'].dt.year
full_data['bulan'] = full_data['tanggal_pencairan'].dt.month
full_data['hari'] = full_data['tanggal_pencairan'].dt.day
full_data['hari_minggu'] = full_data['tanggal_pencairan'].dt.dayofweek
full_data['minggu_tahun'] = full_data['tanggal_pencairan'].dt.isocalendar().week.astype(int)
full_data['kuartal'] = full_data['tanggal_pencairan'].dt.quarter
full_data['awal_bulan'] = (full_data['hari'] <= 10).astype(int)
full_data['akhir_bulan'] = (full_data['hari'] >= 21).astype(int)
full_data['is_weekend'] = (full_data['hari_minggu'] >= 5).astype(int)
full_data['durasi_pinjaman_hari'] = (pd.to_datetime("2025-01-01") - full_data['tanggal_pencairan']).dt.days

full_data.drop('tanggal_pencairan', axis=1, inplace=True)

# =============================================================================
# FEATURE ENGINEERING 
# =============================================================================

def safe_divide(a, b, default=0):
    return np.where(b != 0, a / b, default)

# Financial ratios (same as original)
full_data['rasio_bunga'] = safe_divide(full_data['total_pengembalian'], full_data['jumlah_pinjaman'])
full_data['beban_harian'] = safe_divide(full_data['total_pengembalian'], full_data['durasi_pinjaman_hari'])
full_data['selisih_pengembalian'] = full_data['total_pengembalian'] - full_data['jumlah_pinjaman']
full_data['suku_bunga_aprox'] = safe_divide(
    full_data['total_pengembalian'] - full_data['jumlah_pinjaman'],
    full_data['jumlah_pinjaman']
)

# Log transforms
full_data['jumlah_pinjaman_log'] = np.log1p(full_data['jumlah_pinjaman'].clip(lower=0))
full_data['total_pengembalian_log'] = np.log1p(full_data['total_pengembalian'].clip(lower=0))

# Polynomial features
full_data['jumlah_pinjaman_sq'] = full_data['jumlah_pinjaman'] ** 2

print(f"Total features: {full_data.shape[1]}")

# =============================================================================
# MISSING VALUE HANDLING 
# =============================================================================

numeric_cols = full_data.select_dtypes(include=['number']).columns.tolist()
categorical_cols = full_data.select_dtypes(include=['object']).columns.tolist()

exclude_cols = ['status_gagal_bayar', 'id_transaksi', 'is_train']
numeric_cols = [c for c in numeric_cols if c not in exclude_cols]

full_data[numeric_cols] = full_data[numeric_cols].replace([np.inf, -np.inf], np.nan)

for col in numeric_cols:
    upper_limit = full_data[col].quantile(0.999)
    lower_limit = full_data[col].quantile(0.001)
    full_data[col] = full_data[col].clip(lower=lower_limit, upper=upper_limit)

imputer_num = SimpleImputer(strategy='median')
full_data[numeric_cols] = imputer_num.fit_transform(full_data[numeric_cols])

# =============================================================================
# CATEGORICAL ENCODING
# =============================================================================

label_encoders = {}
for col in categorical_cols:
    full_data[col] = full_data[col].astype(str)
    le = LabelEncoder()
    full_data[col] = le.fit_transform(full_data[col])
    label_encoders[col] = le

# =============================================================================
# SPLIT DATA (SAME AS ORIGINAL)
# =============================================================================
train_clean = full_data[full_data['is_train'] == 1].copy()
test_clean = full_data[full_data['is_train'] == 0].copy()

train_clean.drop(['is_train', 'id_transaksi'], axis=1, inplace=True)
test_ids = test_clean['id_transaksi']
test_clean.drop(['is_train', 'id_transaksi', 'status_gagal_bayar'], axis=1, inplace=True)

X = train_clean.drop('status_gagal_bayar', axis=1)
y = train_clean['status_gagal_bayar'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training: {X_train.shape[0]}, Validation: {X_val.shape[0]}")

# =============================================================================
# CLASS BALANCING 
# =============================================================================

minority_indices = y_train[y_train == 1].index
majority_count = len(y_train[y_train == 0])

n_minority_needed = int(majority_count * 0.5)
oversampled_indices = np.random.choice(minority_indices, size=n_minority_needed, replace=True)

X_train_balanced = pd.concat([X_train, X_train.loc[oversampled_indices]])
y_train_balanced = pd.concat([y_train, y_train.loc[oversampled_indices]])

print(f"Before: {dict(y_train.value_counts())}")
print(f"After: {dict(y_train_balanced.value_counts())}")

# =============================================================================
# MODEL TRAINING - UPGRADED WITH XGB, LGBM, CATBOOST
# =============================================================================

models = {}
results = {}

# 1. Random Forest 
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf_model.fit(X_train_balanced, y_train_balanced)
models['RandomForest'] = rf_model

# 2. Gradient Boosting 
gb_model = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_split=10,
    random_state=RANDOM_STATE
)
gb_model.fit(X_train_balanced, y_train_balanced)
models['GradientBoosting'] = gb_model

# 3. Extra Trees 
et_model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)
et_model.fit(X_train_balanced, y_train_balanced)
models['ExtraTrees'] = et_model

# 4. XGBoost 
xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.3,
    reg_lambda=1.0,
    min_child_weight=3,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_model.fit(X_train_balanced, y_train_balanced)
models['XGBoost'] = xgb_model

# 5. LightGBM 
lgbm_model = LGBMClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.3,
    reg_lambda=1.0,
    min_child_samples=20,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    verbose=-1
)
lgbm_model.fit(X_train_balanced, y_train_balanced)
models['LightGBM'] = lgbm_model

# 6. CatBoost 
catboost_model = CatBoostClassifier(
    iterations=400,
    depth=7,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    random_state=RANDOM_STATE,
    verbose=False
)
catboost_model.fit(X_train_balanced, y_train_balanced)
models['CatBoost'] = catboost_model

# =============================================================================
# EVALUATE ALL MODELS
# =============================================================================
print("\n" + "-" * 70)
print(f"{'Model':<18} {'Accuracy':<12} {'F1 Macro':<12} {'F1 Class1':<12} {'ROC-AUC':<12}")
print("-" * 70)

for name, model in models.items():
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    
    acc = accuracy_score(y_val, y_pred)
    f1_macro = f1_score(y_val, y_pred, average='macro')
    f1_class1 = f1_score(y_val, y_pred, pos_label=1)
    roc_auc = roc_auc_score(y_val, y_proba)
    
    results[name] = {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_class1': f1_class1,
        'roc_auc': roc_auc
    }
    
    print(f"{name:<18} {acc:<12.4f} {f1_macro:<12.4f} {f1_class1:<12.4f} {roc_auc:<12.4f}")

# =============================================================================
# STACKING ENSEMBLE - UPGRADED 
# =============================================================================
print("\n" + "-" * 70)

estimators = [
    ('rf', models['RandomForest']),
    ('gb', models['GradientBoosting']),
    ('et', models['ExtraTrees']),
    ('xgb', models['XGBoost']),
    ('lgbm', models['LightGBM']),
    ('catboost', models['CatBoost']),
]

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, C=0.5),
    cv=5,
    n_jobs=-1
)

stacking_model.fit(X_train_balanced, y_train_balanced)

y_pred_stack = stacking_model.predict(X_val)
y_proba_stack = stacking_model.predict_proba(X_val)[:, 1]

stack_acc = accuracy_score(y_val, y_pred_stack)
stack_f1 = f1_score(y_val, y_pred_stack, average='macro')
stack_f1_c1 = f1_score(y_val, y_pred_stack, pos_label=1)
stack_auc = roc_auc_score(y_val, y_proba_stack)

print(f"{'Stacking(6)':<18} {stack_acc:<12.4f} {stack_f1:<12.4f} {stack_f1_c1:<12.4f} {stack_auc:<12.4f}")

# =============================================================================
# WEIGHTED BLEND 
# =============================================================================
print("\n" + "-" * 70)

# Get probabilities
probas = {name: model.predict_proba(X_val)[:, 1] for name, model in models.items()}

# Weights based on F1 Macro squared
weights = {name: results[name]['f1_macro'] ** 2 for name in models.keys()}
total_w = sum(weights.values())
weights = {k: v/total_w for k, v in weights.items()}

print("Weights:", {k: f"{v:.3f}" for k, v in sorted(weights.items(), key=lambda x: x[1], reverse=True)})

y_proba_blend = sum(weights[name] * probas[name] for name in models.keys())

# =============================================================================
# THRESHOLD OPTIMIZATION
# =============================================================================

best_threshold_stack = 0.5
best_f1_stack = 0
best_threshold_blend = 0.5
best_f1_blend = 0

thresholds = np.arange(0.15, 0.55, 0.01)

for thresh in thresholds:
    # Stacking
    y_pred_t = (y_proba_stack >= thresh).astype(int)
    f1_t = f1_score(y_val, y_pred_t, average='macro')
    if f1_t > best_f1_stack:
        best_f1_stack = f1_t
        best_threshold_stack = thresh
    
    # Blend
    y_pred_b = (y_proba_blend >= thresh).astype(int)
    f1_b = f1_score(y_val, y_pred_b, average='macro')
    if f1_b > best_f1_blend:
        best_f1_blend = f1_b
        best_threshold_blend = thresh

print(f"Stacking optimal threshold: {best_threshold_stack:.2f} (F1: {best_f1_stack:.4f})")
print(f"Blend optimal threshold: {best_threshold_blend:.2f} (F1: {best_f1_blend:.4f})")

# Choose best approach
if best_f1_stack >= best_f1_blend:
    best_proba = y_proba_stack
    best_threshold = best_threshold_stack
    best_method = "Stacking"
else:
    best_proba = y_proba_blend
    best_threshold = best_threshold_blend
    best_method = "Blend"

print(f"\n→ Best method: {best_method}")

# =============================================================================
# FINAL EVALUATION
# =============================================================================
y_pred_final = (best_proba >= best_threshold).astype(int)

print("\n" + "=" * 70)
print(f"FINAL RESULTS ({best_method})")
print("=" * 70)
print(f"Accuracy:    {accuracy_score(y_val, y_pred_final):.4f}")
print(f"F1 Macro:    {f1_score(y_val, y_pred_final, average='macro'):.4f}")
print(f"F1 Class 1:  {f1_score(y_val, y_pred_final, pos_label=1):.4f}")
print(f"ROC-AUC:     {roc_auc_score(y_val, best_proba):.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred_final))

print("Confusion Matrix:")
cm = confusion_matrix(y_val, y_pred_final)
print(cm)
print(f"\n→ Detected: {cm[1,1]}/201 defaults")
print(f"→ Missed: {cm[1,0]} defaults")

# =============================================================================
# CROSS-VALIDATION CHECK
# =============================================================================
print("\n" + "=" * 70)
print("CROSS-VALIDATION CHECK")
print("=" * 70)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_xgb = cross_val_score(models['XGBoost'], X, y, cv=skf, scoring='f1_macro')
cv_lgbm = cross_val_score(models['LightGBM'], X, y, cv=skf, scoring='f1_macro')
cv_catboost = cross_val_score(models['CatBoost'], X, y, cv=skf, scoring='f1_macro')
cv_gb = cross_val_score(models['GradientBoosting'], X, y, cv=skf, scoring='f1_macro')

print(f"XGBoost CV:          {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}")
print(f"LightGBM CV:         {cv_lgbm.mean():.4f} ± {cv_lgbm.std():.4f}")
print(f"CatBoost CV:         {cv_catboost.mean():.4f} ± {cv_catboost.std():.4f}")
print(f"GradientBoosting CV: {cv_gb.mean():.4f} ± {cv_gb.std():.4f}")

# =============================================================================
# FEATURE IMPORTANCE
# =============================================================================
print("\n" + "=" * 70)
print("TOP 15 FEATURES")
print("=" * 70)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(15).to_string(index=False))

# =============================================================================
# GENERATE SUBMISSIONS
# =============================================================================

# Test probabilities
test_proba_stack = stacking_model.predict_proba(test_clean)[:, 1]
test_probas = {name: model.predict_proba(test_clean)[:, 1] for name, model in models.items()}
test_proba_blend = sum(weights[name] * test_probas[name] for name in models.keys())

# 1. Best method submission
if best_method == "Stacking":
    test_proba_best = test_proba_stack
else:
    test_proba_best = test_proba_blend

test_pred_best = (test_proba_best >= best_threshold).astype(int)
submission['status_gagal_bayar'] = test_pred_best
submission.to_csv('submission_enhanced_simple.csv', index=False)
print(f"✓ submission_enhanced_simple.csv ({best_method}) - {dict(pd.Series(test_pred_best).value_counts())}")

# 2. Stacking submission
test_pred_stack = (test_proba_stack >= best_threshold_stack).astype(int)
sub_stack = submission.copy()
sub_stack['status_gagal_bayar'] = test_pred_stack
sub_stack.to_csv('submission_stacking6.csv', index=False)
print(f"✓ submission_stacking6.csv - {dict(pd.Series(test_pred_stack).value_counts())}")

# 3. Blend submission
test_pred_blend = (test_proba_blend >= best_threshold_blend).astype(int)
sub_blend = submission.copy()
sub_blend['status_gagal_bayar'] = test_pred_blend
sub_blend.to_csv('submission_blend6.csv', index=False)
print(f"✓ submission_blend6.csv - {dict(pd.Series(test_pred_blend).value_counts())}")

# 4. Individual best models
for name in ['CatBoost', 'GradientBoosting', 'LightGBM']:
    sub = submission.copy()
    pred = (test_probas[name] >= best_threshold).astype(int)
    sub['status_gagal_bayar'] = pred
    sub.to_csv(f'submission_{name.lower()}.csv', index=False)
    print(f"✓ submission_{name.lower()}.csv - {dict(pd.Series(pred).value_counts())}")

ENHANCED SIMPLE MODEL - UPGRADED WITH XGBOOST, LGBM, CATBOOST

[1/8] Loading Data...
Train shape: (20012, 13)
Test shape: (5003, 12)
Target distribution: {0: np.int64(19006), 1: np.int64(1006)}
Total features: 30
Training: 16009, Validation: 4003
Before: {0: np.int64(15204), 1: np.int64(805)}
After: {0: np.int64(15204), 1: np.int64(8407)}
   [5/7] LightGBM...

----------------------------------------------------------------------
Model              Accuracy     F1 Macro     F1 Class1    ROC-AUC     
----------------------------------------------------------------------
RandomForest       0.9848       0.9180       0.8440       0.9884      
GradientBoosting   0.9865       0.9269       0.8608       0.9871      
ExtraTrees         0.9825       0.9075       0.8241       0.9859      
XGBoost            0.9848       0.9172       0.8424       0.9846      
LightGBM           0.9833       0.9104       0.8295       0.9842      
CatBoost           0.9848       0.9188       0.8456       0.9866     